# CACIE computational validation audit

## tl;dr

This notebook is the reader-facing, fail-closed audit for the canonical CACIE v1.1 artifacts. Every required numerical, sensitivity, figure, detector-validation, and runtime stage must be present and verified before the final success marker is emitted. It audits saved outputs and their byte lineage without recomputing visibility geometry or bootstrap draws.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from paco_observability.notebook_audit import (
    PASS_MARKER,
    REQUIRED_STAGES,
    audit_canonical_artifacts,
)


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "scripts").is_dir():
            return candidate.resolve()
    raise RuntimeError("Run this notebook from inside the paco-observability repository")


REPO_ROOT = find_repo_root()
configured_output_root = Path(os.environ.get("CACIE_OUTPUT_ROOT", "outputs/canonical"))
OUTPUT_ROOT = (
    configured_output_root
    if configured_output_root.is_absolute()
    else REPO_ROOT / configured_output_root
).resolve(strict=True)
print(f"Repository: {REPO_ROOT}")
print(f"Canonical artifact root: {OUTPUT_ROOT}")

## Context & Methods

The audit checks the governed input gate; the 200-frame 9/17/33 validation; post-development 3-of-15 calibration and calibrated strata; a separate 25-frame subset 17/33/65 convergence run; the complete 40,779-frame primary run; primary and block-duration bootstraps; residual ranking; vertical-coordinate, decimation, and partial-scan sensitivities; both publication figures; the training-isolated detector/box validation and source receipts; and stage-local runtime receipts for main validation, subset convergence, and the full record. Validation, strata, and figure lineage receipts plus downstream source-summary and input-Parquet SHA-256 declarations are matched to the current bytes.

### Key assumptions

The 33 x 33 ray--OBB grid is a computational reference within the detected dynamic OBB abstraction, not physical visibility ground truth. The at-least-3-of-15 rule is a post-development choice selected on 100 calibration frames and evaluated on 100 disjoint held-out frames. Full-record observability is a ratio of sums, and uncertainty uses circular moving blocks of 1,200 consecutive retained-frame observations (nominally about 120 seconds at 10 Hz; source-index gaps can extend the elapsed span).

In [ ]:
artifact_audit = audit_canonical_artifacts(
    repo_root=REPO_ROOT,
    output_root=OUTPUT_ROOT,
)
stage_table = pd.DataFrame(artifact_audit["stages"])
check_table = pd.DataFrame(artifact_audit["checks"])
file_table = pd.DataFrame(artifact_audit["audited_files"])
display(stage_table)
display(check_table)
display(file_table[["name", "path", "size_bytes", "sha256"]])

## Takeaways

The final cell independently enforces the complete required-stage set and the `VERIFIED` state. Any missing, stale, internally inconsistent, or lineage-mismatched artifact raises an exception before success can be printed.

In [ ]:
observed_stages = tuple(stage_table["stage"].tolist())
nonverified_stages = stage_table.loc[
    ~stage_table["status"].eq("VERIFIED"),
    ["stage", "status"],
].to_dict(orient="records")
if len(observed_stages) != len(REQUIRED_STAGES) or set(observed_stages) != set(REQUIRED_STAGES):
    raise RuntimeError(
        f"Required audit stages differ from observed stages: required={list(REQUIRED_STAGES)}, "
        f"observed={list(observed_stages)}"
    )
if nonverified_stages:
    raise RuntimeError(f"Required audit stages are not VERIFIED: {nonverified_stages}")
if artifact_audit["status"] != "PASS":
    raise RuntimeError(f"Artifact audit status is not PASS: {artifact_audit['status']}")
display(
    Markdown(
        f"**Execution conclusion:** all {len(REQUIRED_STAGES)} required stages were VERIFIED; "
        f"{artifact_audit['check_count']} fail-closed checks passed across "
        f"{len(artifact_audit['audited_files'])} content-hashed files. "
        "Interpretation remains limited to the declared dynamic-box abstraction and "
        "training-isolated internal detector validation."
    )
)
print(PASS_MARKER)